In [1]:
import torch
from experiments.dj.result_tables import (
    AdaptPriorResult,
    FlowPriorResult,
    LikelihoodResult,
)
from task_transfer.evaluation.evaluate_generative_model import (
    evaluate_flow_prior,
    evaluate_generative_model,
)
from task_transfer.ml_lib.data_loading import build_dataloaders
from task_transfer.ml_lib.trainer_building import build_prior_adapt_trainer

import datajoint as dj
from experiments.dj.dataloader_tables import DataLoaderConfig, AltDataLoaderConfig
from experiments.dj.likelihood_tables import LikelihoodConfig
from experiments.dj.posterior_tables import SBVGPConfig
from experiments.dj.prior_tables import FlowPriorConfig, AdaptPriorConfig
from experiments.dj.sysident_tables import SIConfig
from experiments.dj.result_tables import (
    FlowPriorResult,
    FPSamples,
    FPSamplesConfig,
    LikelihoodResult,
    MLPCondSamples,
    MLPCondSamplesConfig,
    SBVGPResult,
    SIResult,
    AdaptPriorResult,
)
from experiments.dj.evaluation_tables import SIEval, SBVGPEval
from experiments.dj.trainer_tables import (
    FPTrainerConfig,
    LLTrainerConfig,
    SBVGPTrainerConfig,
    SITrainerConfig,
    AdaptPriorTrainer,
)
from experiments.dj.dj_helpers import fetch_best_model_results
from task_transfer.ml_lib.data_loading import build_dataloaders

[2024-07-08 06:06:45,711][INFO]: Connecting sshrinivasan@134.76.19.44:3306
[2024-07-08 06:06:47,298][INFO]: Connected sshrinivasan@134.76.19.44:3306


In [2]:
seed = 42
torch.manual_seed(seed)

In [3]:
AdaptPriorTrainer()

id,mc_sample_size,lr,weight_decay,n_epochs,batch_size,early_stopping_threshold,early_stopping_patience
6a40e4f70ae9e60ad158a5da6db1661d,100,0.001,0.001,300,128,1000,1000
e267b2071bca2c3f9431f155e8e58f23,50000,0.001,0.001,300,128,1000,1000


In [4]:
AdaptPriorConfig()

seed,prior_fp_id to index into FlowPriorConfig,prior_trainer_id to index into FPTrainerConfig,likelihood_id to index into LikelihoodConfig,likelihood_trainer_id to index into LLTrainerConfig,orig_dl_id to index into DataLoaderConfig used for the prior and likelihood training
-100,d0cf491f03b7f839c8a54834a6168081,c40a50ce9c77369770dddd5129836477,a67b8eaff13e89e7272e90768c2ab280,f89651063b51487dcdf4041336ef89db,260a5ea8175f75eaef132f42873ad14a
42,89c1053a65023b042dc63f7f852bb5b0,f89651063b51487dcdf4041336ef89db,a67b8eaff13e89e7272e90768c2ab280,f89651063b51487dcdf4041336ef89db,260a5ea8175f75eaef132f42873ad14a
42,d0cf491f03b7f839c8a54834a6168081,c40a50ce9c77369770dddd5129836477,a67b8eaff13e89e7272e90768c2ab280,f89651063b51487dcdf4041336ef89db,260a5ea8175f75eaef132f42873ad14a
100,89c1053a65023b042dc63f7f852bb5b0,f89651063b51487dcdf4041336ef89db,a67b8eaff13e89e7272e90768c2ab280,f89651063b51487dcdf4041336ef89db,260a5ea8175f75eaef132f42873ad14a
100,d0cf491f03b7f839c8a54834a6168081,c40a50ce9c77369770dddd5129836477,a67b8eaff13e89e7272e90768c2ab280,f89651063b51487dcdf4041336ef89db,260a5ea8175f75eaef132f42873ad14a


In [5]:
AdaptPriorResult()

seed,prior_fp_id to index into FlowPriorConfig,prior_trainer_id to index into FPTrainerConfig,likelihood_id to index into LikelihoodConfig,likelihood_trainer_id to index into LLTrainerConfig,orig_dl_id to index into DataLoaderConfig used for the prior and likelihood training,trainer_id,dl_id,"train_marginal_obs_ll_mean mean per trial, per sample, in nats",train_marginal_obs_ll_sem standard error of the mean,val_marginal_obs_ll_mean,val_marginal_obs_ll_sem,test_marginal_obs_ll_mean,test_marginal_obs_ll_sem,"train_prior_ll_mean mean per trial, per sample, in nats",train_prior_ll_sem standard error of the mean,val_prior_ll_mean,val_prior_ll_sem,test_prior_ll_mean,test_prior_ll_sem,tracker_output,eval_output,model trained joint model NOT just the prior
-100,d0cf491f03b7f839c8a54834a6168081,c40a50ce9c77369770dddd5129836477,a67b8eaff13e89e7272e90768c2ab280,f89651063b51487dcdf4041336ef89db,260a5ea8175f75eaef132f42873ad14a,e267b2071bca2c3f9431f155e8e58f23,260a5ea8175f75eaef132f42873ad14a,-5339.22021484375,623.21923828125,-5353.14404296875,1219.138671875,-5373.67822265625,1560.8585205078125,-225.44107055664062,3.2020812034606934,-225.2079620361328,5.9614667892456055,-224.12705993652344,8.175251007080078,=BLOB=,=BLOB=,=BLOB=
-100,d0cf491f03b7f839c8a54834a6168081,c40a50ce9c77369770dddd5129836477,a67b8eaff13e89e7272e90768c2ab280,f89651063b51487dcdf4041336ef89db,260a5ea8175f75eaef132f42873ad14a,e267b2071bca2c3f9431f155e8e58f23,9ef3ae6fea33eba634d928a88b866836,-5203.6923828125,593.451171875,-5200.5927734375,1111.11376953125,-5184.78564453125,1497.9891357421875,-225.7065887451172,3.2274301052093506,-225.50997924804688,6.009249210357666,-224.40745544433594,8.13746452331543,=BLOB=,=BLOB=,=BLOB=
42,89c1053a65023b042dc63f7f852bb5b0,f89651063b51487dcdf4041336ef89db,a67b8eaff13e89e7272e90768c2ab280,f89651063b51487dcdf4041336ef89db,260a5ea8175f75eaef132f42873ad14a,e267b2071bca2c3f9431f155e8e58f23,260a5ea8175f75eaef132f42873ad14a,-4452.72412109375,501.619140625,-4446.9677734375,931.7738647460938,-4504.203125,1262.531982421875,-4787.8984375,381.42791748046875,-4756.22119140625,715.9622192382812,-4610.33740234375,933.390625,=BLOB=,=BLOB=,=BLOB=
42,d0cf491f03b7f839c8a54834a6168081,c40a50ce9c77369770dddd5129836477,a67b8eaff13e89e7272e90768c2ab280,f89651063b51487dcdf4041336ef89db,260a5ea8175f75eaef132f42873ad14a,6a40e4f70ae9e60ad158a5da6db1661d,260a5ea8175f75eaef132f42873ad14a,-5508.91650390625,709.6250610351562,-5511.12744140625,1374.79931640625,-5575.3623046875,1756.187255859375,-189.88673400878906,2.256335973739624,-189.78884887695312,4.225911617279053,-189.56430053710938,6.074876308441162,=BLOB=,=BLOB=,=BLOB=
42,d0cf491f03b7f839c8a54834a6168081,c40a50ce9c77369770dddd5129836477,a67b8eaff13e89e7272e90768c2ab280,f89651063b51487dcdf4041336ef89db,260a5ea8175f75eaef132f42873ad14a,e267b2071bca2c3f9431f155e8e58f23,9ef3ae6fea33eba634d928a88b866836,-4989.09033203125,581.8089599609375,-4902.71240234375,996.241455078125,-4779.85595703125,1295.3453369140625,-123.60945892333984,2.029911994934082,-123.82306671142578,3.7798922061920166,-123.90569305419922,5.306024074554443,=BLOB=,=BLOB=,=BLOB=


In [14]:
DataLoaderConfig()

id,data_fname,train_prop,val_prop
260a5ea8175f75eaef132f42873ad14a,/src/project/data/synthetic/haefner_2afc/original_haefner_2afc_task_1_dataset.pkl,0.7,0.2


In [16]:
AltDataLoaderConfig()

id,data_fname,train_prop,val_prop
260a5ea8175f75eaef132f42873ad14a,/src/project/data/synthetic/haefner_2afc/original_haefner_2afc_task_1_dataset.pkl,0.7,0.2
9ef3ae6fea33eba634d928a88b866836,/src/project/data/synthetic/haefner_2afc/original_haefner_2afc_task_2_dataset.pkl,0.7,0.2


In [7]:
prior_config_proj_col = "fp_id"

In [8]:
download_path = "/tmp"
criterion = "val_ll_mean"
k = 1
flow_prior_config_table = FlowPriorConfig 
best_val_prior_results = fetch_best_model_results(
    result_table=FlowPriorResult,
    config_table=flow_prior_config_table,
    data_loader_config_table=DataLoaderConfig,
    trainer_config_table=FPTrainerConfig,
    config_proj_col=prior_config_proj_col,
    criterion=criterion,
    k=k,
    download_path=download_path,
)

In [9]:
flow_model = torch.load(
    best_val_prior_results["model"], map_location="cpu"
)

In [10]:
flow_model

FlowDistribution(
  (base_distribution): TrainableDistributionAdapter(
    distribution_class=<class 'torch.distributions.multivariate_normal.MultivariateNormal'>, loc=tensor([0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
            0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.]), covariance_matrix=Covariance()
    (covariance_matrix): Covariance()
  )
  (transform): SequentialTransform(
    (transforms): ModuleList(
      (0): InverseTransform(
        (transform): Softplus()
      )
      (1): IndependentAffine()
      (2): Tanh()
      (3): IndependentAffine()
      (4): Tanh()
      (5): IndependentAffine()
    )
  )
)

In [11]:
flow_model.base_distribution

TrainableDistributionAdapter(
  distribution_class=<class 'torch.distributions.multivariate_normal.MultivariateNormal'>, loc=tensor([0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
          0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.]), covariance_matrix=Covariance()
  (covariance_matrix): Covariance()
)